In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import dill

In [60]:
class BPE:
    def __init__(self, vocab_size: int):
        self.vocab_size = vocab_size
        self.id2token = None
        self.token2id = None
    
    def fit(self, text: str):
        uniq_tokens = sorted(set(text))
        tokens_list = list(text)

        while len(uniq_tokens) < self.vocab_size:
            print(len(uniq_tokens), '|', self.vocab_size)
            pairs = {}

            for i in range(len(tokens_list)-1):
                if (tokens_list[i], tokens_list[i+1]) not in pairs:
                    pairs[(tokens_list[i], tokens_list[i+1])] = 0
                pairs[(tokens_list[i], tokens_list[i+1])] += 1
            
            if len(pairs) == 0: break

            top_freq = 0
            top_pair = None
            for pair, freq in pairs.items():
                if freq > top_freq:
                    top_freq = freq
                    top_pair = pair
            
            uniq_tokens.append(top_pair[0]+top_pair[1])

            upd_tokens_list = []
            i=0
            while i < len(tokens_list):
                if (i < len(tokens_list) - 1 and tokens_list[i:i+2] == list(top_pair)):
                    upd_tokens_list.append(top_pair[0]+top_pair[1])
                    i+=2
                else:
                    upd_tokens_list.append(tokens_list[i])
                    i+=1

            tokens_list = upd_tokens_list
        
        self.id2token = {i: token for i, token in enumerate(uniq_tokens[:self.vocab_size])}
        self.token2id = {token: i for i, token in self.id2token.items()}

    def encode(self, text: str) -> list[int]:

        tokens_list = list(text)

        fin_flag = True
        while fin_flag:
            fin_flag = False

            upd_tokens_list = []
            i=0
            while i < len(tokens_list):
                if (i < len(tokens_list) - 1 and tokens_list[i]+tokens_list[i+1] in self.token2id):
                    upd_tokens_list.append(tokens_list[i]+tokens_list[i+1])
                    i+=2
                    fin_flag = True
                else:
                    upd_tokens_list.append(tokens_list[i])
                    i+=1
            
            tokens_list = upd_tokens_list
        
        token_id_list = []

        for token in tokens_list:
            token_id_list.append(self.token2id[token])
        
        return(token_id_list)
    
    def decode(self, token_ids: list[int]) -> str:

        return ''.join(self.id2token[i] for i in token_ids)

    def save(self, filename):
        with open(filename, 'wb') as f:
            dill.dump(self, f)
        print(f"Объект сохранён в {filename}")

    @classmethod
    def load(cls, filename):
        with open(filename, 'rb') as f:
            obj = dill.load(f)
                
        print(f"Объект загружен из {filename}")
        return obj


In [61]:
drova = "На дворе дрова, за двором дрова, дрова вширь двора, не вместит двор дров, надо дрова выдворить на дровяной двор."

In [62]:
bpe = BPE(28)
bpe.fit(drova)

print(bpe.id2token)
print(bpe.encode('вором дрова, дрова вширь двора, не вмест'))
print(bpe.decode(bpe.encode('вором дрова, дрова вширь двора, не вмест')))

21 | 28
22 | 28
23 | 28
24 | 28
25 | 28
26 | 28
27 | 28
{0: ' ', 1: ',', 2: '.', 3: 'Н', 4: 'а', 5: 'в', 6: 'д', 7: 'е', 8: 'з', 9: 'и', 10: 'й', 11: 'м', 12: 'н', 13: 'о', 14: 'р', 15: 'с', 16: 'т', 17: 'ш', 18: 'ы', 19: 'ь', 20: 'я', 21: ' д', 22: 'ро', 23: 'во', 24: ' дро', 25: ' дров', 26: ' дво', 27: ' двор'}
[23, 22, 11, 25, 4, 1, 25, 4, 0, 5, 17, 9, 14, 19, 27, 4, 1, 0, 12, 7, 0, 5, 11, 7, 15, 16]
вором дрова, дрова вширь двора, не вмест
